# 09b — Local model API contracts — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/09.md) · [Course map](../PLAN.md) · [Project](../../projects/stage09/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Build request/response adapters
- Apply explicit timeouts and output limits
- Benchmark real model calls separately from offline checks


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

A model server exposes inference through a protocol. Ollama’s native chat API and a vLLM OpenAI-compatible endpoint use different request and response shapes. Keep an adapter boundary so orchestration does not depend on one wire format. Configure base URL, model ID, limits, and timeouts explicitly.

A non-streaming request is easier to inspect first; streaming reduces perceived latency but requires incremental parsing and cancellation. Model IDs, weights, licenses, hardware requirements, and endpoint behavior change. Read the server’s current docs and use a model installed on your machine. The offline notebook validates payload contracts only; the project performs actual localhost HTTP calls and records observed timing.

Private hosting still requires access control, secure networking, data retention policy, updates, and monitoring. Do not expose an unauthenticated inference port to the internet. On macOS, start with a server supported by the machine; the vLLM exercise belongs on a supported Linux accelerator host.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import json
payload = {"model": "configured-local-model", "messages": [{"role": "user", "content": "Hello"}], "stream": False}
print(json.dumps(payload))


## Exercise 1: Ollama request

Return a native chat payload with model, one user message, stream=False, and options.num_predict=max_tokens. Require 1..1024 tokens and nonempty prompt <=4000 characters.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def ollama_payload(model, prompt, max_tokens=128):
    if not prompt.strip() or len(prompt) > 4000 or not 1 <= max_tokens <= 1024: raise ValueError("Request bounds")
    return {"model": model, "messages": [{"role": "user", "content": prompt}], "stream": False, "options": {"num_predict": max_tokens}}


In [ ]:
assert ollama_payload("demo", "hello")["options"]["num_predict"] == 128
expect_error(ValueError, lambda: ollama_payload("demo", "x"*4001))
print("Exercise 1: checks passed")


**Why this works:** Character limits are a coarse guard; production admission must also count actual model tokens.


## Exercise 2: Response parsing

Extract message.content as a string from a native Ollama response; malformed responses raise ValueError.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def parse_ollama(body):
    try: content = body["message"]["content"]
    except (KeyError, TypeError): raise ValueError("Malformed response") from None
    if not isinstance(content, str): raise ValueError("Malformed content")
    return content


In [ ]:
assert parse_ollama({"message": {"content": "hello"}}) == "hello"
expect_error(ValueError, lambda: parse_ollama({"error": "model missing"}))
print("Exercise 2: checks passed")


**Why this works:** Failures should remain visible for retries, metrics, and user communication.


## Exercise 3: Throughput

Return tokens/second from token counts and total elapsed wall seconds; reject nonpositive elapsed.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def throughput(token_counts, elapsed):
    if elapsed <= 0: raise ValueError("Positive elapsed required")
    return sum(token_counts)/elapsed


In [ ]:
assert throughput([10, 20], 2) == 15
expect_error(ValueError, lambda: throughput([1], 0))
print("Exercise 3: checks passed")


**Why this works:** Adding per-request durations would misrepresent concurrent server throughput.


## Independent transfer

Run projects/stage09/serve.py against an installed model. Save at least 30 warm and cold observations, output lengths, failures, p50/p95, and machine details. Repeat on vLLM only on compatible hardware.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. What does an offline adapter test prove?

   Request/response handling, not model availability, answer quality, or actual serving performance.

2. Why cap output tokens?

   To bound inference work and make latency/cost more predictable.


## Reading and review

- [Primary reference 1](https://docs.ollama.com/api/chat)
- [Primary reference 2](https://docs.vllm.ai/en/latest/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
